# <center>🏥 EpiForecast-MX — Avance 3: Modelo Baseline</center>

<center>

**Maestría en Inteligencia Artificial Aplicada — Tecnológico de Monterrey**

TC5035 — Proyecto Integrador

---

**Pronóstico Epidemiológico de Padecimientos Neurológicos y de Salud Mental en México**

Depresión (F32) · Parkinson (G20) · Alzheimer (G30)

---

| Rol | Nombre | Institución |
|---|---|---|
| Estudiante — Desarrollo | Javier Augusto Rebull Saucedo | Tec de Monterrey / Santander US |
| Estudiante — Desarrollo | Juan Carlos Pérez Nava | Tec de Monterrey / IMSS |
| Estudiante — Desarrollo | Luis Gerardo Sánchez Salazar | Tec de Monterrey / Tesla |
| Asesora Académica | Dra. Grettel Barceló Alonso | Tecnológico de Monterrey |
| Sponsor / Líder de Proyecto | Dra. Ruth Pérez | IMSS |
| Investigadora en Psiquiatría | Dra. Lina Díaz Castro | IMSS |

**Equipo 01 · Semana 05 · Febrero 2026**

</center>

---
## Tabla de Contenidos

1. [Configuración del Entorno](#1-configuración-del-entorno)
2. [Contexto y Justificación del Baseline](#2-contexto-y-justificación-del-baseline)
3. [Carga y Preparación de Datos](#3-carga-y-preparación-de-datos)
4. [Definición de Métricas de Desempeño](#4-definición-de-métricas-de-desempeño)
5. [Modelo Baseline — Nivel Nacional](#5-modelo-baseline--nivel-nacional)
6. [Modelo Baseline — Nacional por Sexo](#6-modelo-baseline--nacional-por-sexo)
7. [Modelo Baseline — Por Entidad Federativa](#7-modelo-baseline--por-entidad-federativa)
8. [Modelo Baseline — Por Entidad y Sexo](#8-modelo-baseline--por-entidad-y-sexo)
9. [Análisis de Sub/Sobreajuste](#9-análisis-de-subsobreajuste)
10. [Análisis de Componentes (Importancia de Características)](#10-análisis-de-componentes-importancia-de-características)
11. [Resumen Consolidado de Métricas](#11-resumen-consolidado-de-métricas)
12. [Desempeño Mínimo Aceptable](#12-desempeño-mínimo-aceptable)
13. [Conclusiones y Siguientes Pasos](#13-conclusiones-y-siguientes-pasos)
14. [Referencias](#14-referencias)
---

## 1. Configuración del Entorno <a id='1-configuración-del-entorno'></a>

Se importan las bibliotecas necesarias y se establece la configuración visual institucional del IMSS, consistente con los entregables anteriores del proyecto.

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
from pathlib import Path
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_plotly, plot_components_plotly

import warnings
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import numpy as np
import seaborn as sns
from loguru import logger
import sys
import logging

# Suprimir logs verbosos de cmdstanpy y Prophet
logging.getLogger("cmdstanpy").disabled = True
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*Disabling yearly.*")
warnings.filterwarnings("ignore", message=".*Disabling weekly.*")

# Configuración de logger
logger.remove()
logger.add(sys.stderr, level="INFO", format="{time:HH:mm:ss} | {level:<7} | {message}")
logger.info("Avance 3 — EpiForecast-MX Baseline inicializado")

In [ ]:
# =============================================================================
# CONSTANTES Y CONFIGURACIÓN
# =============================================================================

# --- Rutas -------------------------------------------------------------------
DATA_PATH = Path("../data/processed/data_inegi_General.csv")

# --- Paleta IMSS institucional (Guía cromática oficial) ----------------------
IMSS_COLORS = {
    "neutral_black":  "#231F20",  # PANTONE Neutral Black C
    "burgundy":       "#9B2242",  # PANTONE 7420 C
    "dark_burgundy":  "#6F1D46",  # PANTONE 7421 C
    "cool_gray":      "#97999B",  # PANTONE Cool Gray C
    "teal":           "#00524E",  # PANTONE IMSS 561 C
    "dark_teal":      "#173F35",  # PANTONE 627 C
    "cream":          "#E8D5B5",  # PANTONE 7402 C
    "gold":           "#B58500",  # PANTONE 1255 C
}

# Paleta secuencial para gráficos
PALETTE_MAIN = [
    IMSS_COLORS["teal"],
    IMSS_COLORS["burgundy"],
    IMSS_COLORS["gold"],
    IMSS_COLORS["dark_teal"],
    IMSS_COLORS["dark_burgundy"],
    IMSS_COLORS["cool_gray"],
    IMSS_COLORS["neutral_black"],
    IMSS_COLORS["cream"],
]

PALETTE_PADECIMIENTO = {
    "Depresión":  IMSS_COLORS["burgundy"],
    "Parkinson":  IMSS_COLORS["teal"],
    "Alzheimer":  IMSS_COLORS["gold"],
}

PALETTE_SEXO = {
    "Hombres": IMSS_COLORS["teal"],
    "Mujeres": IMSS_COLORS["burgundy"],
}

# --- Parámetros del Baseline --------------------------------------------------
HORIZONTE_SEMANAS    = 84          # ~1.6 años de pronóstico
CV_INITIAL           = '730 days'  # Período inicial de entrenamiento (~2 años)
CV_PERIOD            = '56 days'   # Intervalo entre cortes de validación (~8 semanas)
CV_HORIZON           = '168 days'  # Horizonte de pronóstico en CV (~24 semanas)

# --- Estilo global de matplotlib ---------------------------------------------
plt.rcParams.update({
    "figure.facecolor":    "white",
    "axes.facecolor":      "white",
    "axes.edgecolor":      IMSS_COLORS["cool_gray"],
    "axes.labelcolor":     IMSS_COLORS["neutral_black"],
    "text.color":          IMSS_COLORS["neutral_black"],
    "xtick.color":         IMSS_COLORS["neutral_black"],
    "ytick.color":         IMSS_COLORS["neutral_black"],
    "axes.grid":           True,
    "grid.alpha":          0.3,
    "grid.color":          IMSS_COLORS["cool_gray"],
    "font.family":         "sans-serif",
    "font.size":           11,
    "axes.titlesize":      13,
    "axes.titleweight":    "bold",
    "figure.titlesize":    15,
    "figure.titleweight":  "bold",
    "figure.dpi":          120,
    "savefig.dpi":         150,
    "savefig.bbox":        "tight",
})

logger.success(f"Configuración cargada | Paleta IMSS: {len(IMSS_COLORS)} colores | Horizonte: {HORIZONTE_SEMANAS} semanas")

---
## 2. Contexto y Justificación del Baseline <a id='2-contexto-y-justificación-del-baseline'></a>

### 2.1 Objetivo del Modelo Baseline

El propósito de este avance es construir un **modelo de referencia (baseline)** que permita:

1. **Evaluar la viabilidad** del problema de pronóstico epidemiológico.
2. **Establecer un piso de desempeño** contra el cual se compararán modelos optimizados (tuneados) en fases posteriores.
3. **Gestionar expectativas** con los stakeholders del IMSS respecto a lo alcanzable con métodos fundamentales.
4. **Validar el pipeline de datos** confirmando que la integración SINAVE–INEGI produce series de tiempo modelables.

### 2.2 ¿Por qué Prophet como Algoritmo Baseline?

La selección de **Prophet** (Taylor & Letham, 2018) como algoritmo baseline se fundamenta en los siguientes criterios, alineados con la metodología CRISP-ML(Q) (Studer et al., 2021):

| Criterio CRISP-ML(Q) | Justificación |
|---|---|
| **Tipo de datos** | Series de tiempo epidemiológicas semanales con estacionalidad anual y tendencia a largo plazo. Prophet está diseñado específicamente para este tipo de datos (Taylor & Letham, 2018). |
| **Robustez** | Maneja de forma nativa valores faltantes, outliers y cambios de tendencia (*changepoints*), lo cual es crítico dado el efecto COVID-19 en nuestras series. |
| **Escalabilidad** | Permite parametrizar el pipeline para ejecutar modelos por entidad, por sexo y por padecimiento sin modificar la arquitectura base. |
| **Interpretabilidad** | Descompone la serie en componentes interpretables (tendencia, estacionalidad), facilitando la comunicación de resultados a las doctoras del IMSS. |
| **Intervalos de predicción** | Genera intervalos de incertidumbre (*uncertainty intervals*) de forma nativa, requisito fundamental para la planificación estratégica en salud pública. |
| **Evidencia previa** | En la Fase I del proyecto, se demostró que modelos de ML recursivos (XGBoost, Random Forest) presentan **acumulación progresiva de error** en horizontes largos (52+ semanas), mientras que Prophet mantiene un desempeño más estable. |

### 2.3 Descarte de Modelos Alternativos

Se descartaron los siguientes enfoques para el baseline por razones específicas:

- **XGBoost / Random Forest recursivo**: La Dra. Grettel identificó *data leakage* en la implementación previa. Adicionalmente, la acumulación progresiva de error en pronósticos de horizonte largo (84 semanas) los hace inadecuados como baseline robusto.
- **ARIMA / SARIMA**: Requieren estacionariedad estricta y la especificación manual de órdenes (p, d, q). No manejan de forma nativa los changepoints por COVID-19.
- **Modelo naïve (última observación repetida)**: Si bien es un baseline válido en la literatura, no captura la estacionalidad anual que es un patrón documentado en enfermedades neurológicas y de salud mental.

### 2.4 Niveles de Modelado

Siguiendo la directriz de las doctoras del IMSS y la asesora académica, se construirán modelos baseline en **cuatro niveles de granularidad**:

| Nivel | Descripción | Propósito |
|---|---|---|
| **Nacional** | Serie agregada de los 32 estados | Visión macro del comportamiento epidemiológico |
| **Nacional por Sexo** | Serie nacional desagregada Hombres/Mujeres | Identificar diferencias de género en la incidencia |
| **Por Entidad Federativa** | Un modelo por cada estado | Pronóstico estatal para asignación de recursos |
| **Por Entidad y Sexo** | Un modelo por estado × sexo | Máxima granularidad para planificación operativa |

---
## 3. Carga y Preparación de Datos <a id='3-carga-y-preparación-de-datos'></a>

Se carga el dataset resultante del pipeline de limpieza y transformación (Avances 1 y 2), que integra datos del boletín epidemiológico de SINAVE con datos demográficos de INEGI.

In [ ]:
# =============================================================================
# CARGA DE DATOS
# =============================================================================
df_datos = pd.read_csv(DATA_PATH)
df_datos['Fecha'] = pd.to_datetime(df_datos['Fecha'])

logger.info(f"Dataset cargado: {df_datos.shape[0]:,} registros × {df_datos.shape[1]} columnas")
logger.info(f"Rango temporal: {df_datos['Fecha'].min().strftime('%Y-%m-%d')} → {df_datos['Fecha'].max().strftime('%Y-%m-%d')}")
df_datos.info()

In [ ]:
df_datos.head(10)

### 3.1 Preparación de Series para Prophet

Prophet requiere un DataFrame con columnas `ds` (fecha) y `y` (valor a pronosticar). Se construyen las series a los cuatro niveles de granularidad.

In [ ]:
# =============================================================================
# SERIE NIVEL NACIONAL (agregado de todos los estados)
# =============================================================================
serie_nacional = (
    df_datos
    .groupby(["Fecha"])[["incrementos_hombres", "incrementos_mujeres"]]
    .sum()
    .reset_index()
    .rename(columns={'Fecha': 'ds'})
)
serie_nacional["y"] = serie_nacional["incrementos_hombres"] + serie_nacional["incrementos_mujeres"]
serie_nacional = serie_nacional.sort_values('ds').reset_index(drop=True)

logger.info(f"Serie Nacional: {len(serie_nacional)} observaciones semanales")
serie_nacional.head()

In [ ]:
# =============================================================================
# SERIE NIVEL NACIONAL POR SEXO
# =============================================================================
serie_nac_hombres = (
    df_datos
    .groupby(["Fecha"])["incrementos_hombres"]
    .sum()
    .reset_index()
    .rename(columns={'Fecha': 'ds', 'incrementos_hombres': 'y'})
    .sort_values('ds')
    .reset_index(drop=True)
)

serie_nac_mujeres = (
    df_datos
    .groupby(["Fecha"])["incrementos_mujeres"]
    .sum()
    .reset_index()
    .rename(columns={'Fecha': 'ds', 'incrementos_mujeres': 'y'})
    .sort_values('ds')
    .reset_index(drop=True)
)

logger.info(f"Serie Nacional Hombres: {len(serie_nac_hombres)} obs | Mujeres: {len(serie_nac_mujeres)} obs")

In [ ]:
# =============================================================================
# SERIE POR ENTIDAD FEDERATIVA
# =============================================================================
serie_estados = (
    df_datos
    .groupby(["Fecha", "Entidad"])[["incrementos_hombres", "incrementos_mujeres"]]
    .sum()
    .reset_index()
    .rename(columns={'Fecha': 'ds'})
)
serie_estados["y"] = serie_estados["incrementos_hombres"] + serie_estados["incrementos_mujeres"]
serie_estados = serie_estados.sort_values(['Entidad', 'ds']).reset_index(drop=True)

entidades = sorted(serie_estados['Entidad'].unique())
logger.info(f"Series por Entidad: {len(entidades)} estados | {len(serie_estados):,} registros totales")
print(f"Entidades: {entidades}")

In [ ]:
# =============================================================================
# SERIE POR ENTIDAD FEDERATIVA Y SEXO
# =============================================================================
# Se crean DataFrames separados para hombres y mujeres por estado
serie_estados_hombres = (
    df_datos
    .groupby(["Fecha", "Entidad"])["incrementos_hombres"]
    .sum()
    .reset_index()
    .rename(columns={'Fecha': 'ds', 'incrementos_hombres': 'y'})
    .sort_values(['Entidad', 'ds'])
    .reset_index(drop=True)
)

serie_estados_mujeres = (
    df_datos
    .groupby(["Fecha", "Entidad"])["incrementos_mujeres"]
    .sum()
    .reset_index()
    .rename(columns={'Fecha': 'ds', 'incrementos_mujeres': 'y'})
    .sort_values(['Entidad', 'ds'])
    .reset_index(drop=True)
)

logger.info(f"Series Estado×Sexo — Hombres: {len(serie_estados_hombres):,} | Mujeres: {len(serie_estados_mujeres):,}")

---
## 4. Definición de Métricas de Desempeño <a id='4-definición-de-métricas-de-desempeño'></a>

### 4.1 Selección de Métricas

Para un problema de **regresión de series de tiempo** en el contexto epidemiológico, se seleccionan las siguientes métricas de desempeño (Hyndman & Koehler, 2006):

| Métrica | Fórmula | Interpretación | Justificación en contexto IMSS |
|---|---|---|---|
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$ | Error cuadrático medio; penaliza errores grandes | Captura errores extremos que podrían afectar la asignación de recursos |
| **MAE** | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | Error absoluto promedio en unidades originales | Interpretable directamente como "casos de diferencia" promedio |
| **MAPE** | $\frac{1}{n}\sum\left|\frac{y_i - \hat{y}_i}{y_i}\right| \times 100$ | Error porcentual relativo | Permite comparar desempeño entre estados con escalas diferentes |
| **MDAPE** | Mediana de $\left|\frac{y_i - \hat{y}_i}{y_i}\right| \times 100$ | Mediana del error porcentual | Robusta ante outliers, complementa al MAPE |

### 4.2 Métrica Principal

Se define el **MAPE** como métrica principal para la evaluación del baseline, dado que:

1. Permite **comparación entre entidades** con volúmenes de incidencia muy distintos (e.g., Ciudad de México vs. Tlaxcala).
2. Es **interpretable para stakeholders no técnicos**: un MAPE de 25% significa que las predicciones tienen, en promedio, un 25% de error respecto al valor real.
3. Es la métrica recomendada por la literatura en pronóstico epidemiológico (Hyndman & Athanasopoulos, 2021).

> **Nota sobre MAPE**: Se excluyen del cálculo las observaciones donde $y_i = 0$ para evitar divisiones indefinidas. Este caso es relevante en estados con muy bajo reporte, como lo señaló la Dra. Grettel en la reunión del 12 de febrero.

### 4.3 Implementación de Métricas

In [ ]:
# =============================================================================
# FUNCIONES DE MÉTRICAS Y UTILIDADES
# =============================================================================

def calcular_mape(y_true, y_pred):
    """
    Calcula el Mean Absolute Percentage Error (MAPE).
    Excluye observaciones donde y_true == 0 para evitar divisiones indefinidas.
    
    Parámetros
    ----------
    y_true : pd.Series — Valores reales
    y_pred : pd.Series — Valores predichos
    
    Retorna
    -------
    float — MAPE en porcentaje
    """
    den = y_true.replace(0, np.nan)
    mape_series = np.abs((y_true - y_pred) / den) * 100
    return mape_series.mean()


def calcular_rmse(y_true, y_pred):
    """Root Mean Squared Error."""
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def calcular_mae(y_true, y_pred):
    """Mean Absolute Error."""
    return np.mean(np.abs(y_true - y_pred))


def generar_pronostico(modelo, periodos=HORIZONTE_SEMANAS, freq='W'):
    """
    Genera pronóstico futuro a partir de un modelo Prophet ajustado.
    
    Parámetros
    ----------
    modelo   : Prophet — Modelo ajustado
    periodos : int — Semanas a pronosticar (default: 84)
    freq     : str — Frecuencia temporal (default: 'W' semanal)
    
    Retorna
    -------
    pd.DataFrame — DataFrame con pronóstico y componentes
    """
    future = modelo.make_future_dataframe(periods=periodos, freq=freq)
    forecast = modelo.predict(future)
    return forecast


logger.success("Funciones de métricas definidas: RMSE, MAE, MAPE")

In [ ]:
# =============================================================================
# FUNCIONES DE VISUALIZACIÓN CON ESTILO IMSS
# =============================================================================

def graficar_pronostico(modelo, forecast, titulo, color_obs=None, color_pred=None):
    """
    Genera gráfico de pronóstico con estilo institucional IMSS.
    """
    if color_obs is None:
        color_obs = IMSS_COLORS["dark_teal"]
    if color_pred is None:
        color_pred = IMSS_COLORS["burgundy"]
    
    fig = modelo.plot(forecast)
    ax = fig.gca()
    ax.set_title(f"Pronóstico Baseline — {titulo}", fontsize=13, fontweight='bold')
    ax.set_xlabel("Fecha")
    ax.set_ylabel("Incrementos Semanales")
    ax.grid(True, linestyle='--', alpha=0.6)
    
    lineas = ax.get_lines()
    if lineas:
        lineas[0].set_color(color_obs)
        lineas[0].set_markersize(3)
        if len(lineas) > 1:
            lineas[1].set_color(color_pred)
            lineas[1].set_linewidth(2)
    
    for col in ax.collections:
        col.set_facecolor(IMSS_COLORS["cream"])
        col.set_edgecolor(IMSS_COLORS["gold"])
        col.set_alpha(0.4)
    
    plt.tight_layout()
    plt.show()
    plt.close(fig)


def graficar_componentes(modelo, forecast, titulo):
    """
    Genera gráfico de componentes (tendencia + estacionalidad) con estilo IMSS.
    """
    fig2 = modelo.plot_components(forecast)
    fig2.suptitle(f"Componentes — {titulo}", fontsize=13, fontweight='bold', y=1.02)
    
    for ax in fig2.get_axes():
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.set_facecolor("white")
        for line in ax.get_lines():
            line.set_color(IMSS_COLORS["burgundy"])
            line.set_linewidth(2)
        for col in ax.collections:
            col.set_facecolor(IMSS_COLORS["cream"])
            col.set_edgecolor(IMSS_COLORS["gold"])
            col.set_alpha(0.4)
    
    plt.tight_layout()
    plt.show()
    plt.close(fig2)


def graficar_cv_metricas(df_pm, titulo):
    """
    Grafica la evolución de métricas por horizonte de pronóstico (cross-validation).
    Útil para analizar degradación del desempeño a mayor horizonte.
    """
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(f"Métricas por Horizonte de Pronóstico — {titulo}", fontsize=13, fontweight='bold')
    
    metricas_plot = [('rmse', 'RMSE'), ('mae', 'MAE'), ('mdape', 'MDAPE')]
    colores = [IMSS_COLORS["teal"], IMSS_COLORS["burgundy"], IMSS_COLORS["gold"]]
    
    for ax, (col, label), color in zip(axes, metricas_plot, colores):
        if col in df_pm.columns:
            ax.plot(df_pm['horizon'].dt.days, df_pm[col], color=color, linewidth=2)
            ax.set_xlabel("Horizonte (días)")
            ax.set_ylabel(label)
            ax.set_title(label)
            ax.grid(True, linestyle='--', alpha=0.4)
    
    plt.tight_layout()
    plt.show()
    plt.close(fig)


logger.success("Funciones de visualización IMSS definidas")

In [ ]:
# =============================================================================
# FUNCIÓN PRINCIPAL: EVALUAR MODELO BASELINE
# =============================================================================

def evaluar_baseline(serie_prophet, nombre, graficar=True, verbose=True):
    """
    Entrena un modelo Prophet baseline (hiperparámetros por defecto), 
    realiza cross-validation y calcula métricas de desempeño.
    
    Parámetros
    ----------
    serie_prophet : pd.DataFrame — con columnas ['ds', 'y']
    nombre        : str — identificador para logs y gráficos
    graficar      : bool — si se generan gráficos (default: True)
    verbose       : bool — si se imprimen métricas (default: True)
    
    Retorna
    -------
    dict con keys: 'modelo', 'forecast', 'cv', 'metricas_cv', 'resumen'
    """
    # 1. Ajustar modelo Prophet con hiperparámetros por defecto (baseline)
    modelo = Prophet()
    modelo.fit(serie_prophet)
    
    # 2. Generar pronóstico
    forecast = generar_pronostico(modelo)
    
    # 3. Cross-validation
    cv_result = cross_validation(
        modelo,
        initial=CV_INITIAL,
        period=CV_PERIOD,
        horizon=CV_HORIZON
    )
    
    # 4. Métricas de Prophet
    df_pm = performance_metrics(cv_result)
    
    # 5. Resumen de métricas
    metricas_disponibles = [c for c in ['rmse', 'mae', 'mdape'] if c in df_pm.columns]
    resumen = df_pm[metricas_disponibles].mean(numeric_only=True).to_dict()
    resumen['mape'] = calcular_mape(cv_result['y'], cv_result['yhat'])
    resumen['nombre'] = nombre
    
    # 6. Métricas en train (para análisis de sub/sobreajuste)
    forecast_train = forecast[forecast['ds'].isin(serie_prophet['ds'])]
    merged_train = serie_prophet.merge(forecast_train[['ds', 'yhat']], on='ds')
    resumen['rmse_train'] = calcular_rmse(merged_train['y'], merged_train['yhat'])
    resumen['mae_train'] = calcular_mae(merged_train['y'], merged_train['yhat'])
    resumen['mape_train'] = calcular_mape(merged_train['y'], merged_train['yhat'])
    
    if verbose:
        logger.success(f"Métricas CV — {nombre}")
        logger.info(f"  RMSE: {resumen.get('rmse', np.nan):.2f} | MAE: {resumen.get('mae', np.nan):.2f} | "
                     f"MAPE: {resumen['mape']:.2f}% | MDAPE: {resumen.get('mdape', np.nan):.4f}")
        logger.info(f"  Train → RMSE: {resumen['rmse_train']:.2f} | MAE: {resumen['mae_train']:.2f} | "
                     f"MAPE: {resumen['mape_train']:.2f}%")
    
    if graficar:
        graficar_pronostico(modelo, forecast, nombre)
        graficar_componentes(modelo, forecast, nombre)
        graficar_cv_metricas(df_pm, nombre)
    
    return {
        'modelo': modelo,
        'forecast': forecast,
        'cv': cv_result,
        'metricas_cv': df_pm,
        'resumen': resumen
    }


logger.success("Función principal evaluar_baseline() definida")

---
## 5. Modelo Baseline — Nivel Nacional <a id='5-modelo-baseline--nivel-nacional'></a>

Se entrena el primer modelo baseline sobre la **serie agregada nacional** (suma de incrementos semanales de todos los estados y ambos sexos). Este modelo proporciona la visión macro del comportamiento epidemiológico combinado de los tres padecimientos (Depresión F32, Parkinson G20, Alzheimer G30).

In [ ]:
# =============================================================================
# MODELO BASELINE — NIVEL NACIONAL
# =============================================================================
logger.info("═" * 60)
logger.info("MODELO BASELINE — NIVEL NACIONAL")
logger.info("═" * 60)

resultado_nacional = evaluar_baseline(
    serie_prophet=serie_nacional[['ds', 'y']].copy(),
    nombre="Nacional (Agregado)",
    graficar=True
)

### 5.1 Interpretación del Modelo Nacional

El gráfico de pronóstico muestra la serie histórica (puntos) y la predicción del modelo (línea) con su intervalo de incertidumbre (banda sombreada). Los componentes descomponen la serie en:

- **Tendencia**: Captura el comportamiento a largo plazo, incluyendo el impacto del COVID-19 como un *changepoint* significativo.
- **Estacionalidad anual**: Patrón cíclico de 52 semanas que refleja la dinámica estacional de los padecimientos.

---
## 6. Modelo Baseline — Nacional por Sexo <a id='6-modelo-baseline--nacional-por-sexo'></a>

Siguiendo la solicitud de la Dra. Ruth Pérez (reunión del 11 de febrero), se desagregan los modelos por sexo a nivel nacional para identificar diferencias en la dinámica epidemiológica entre hombres y mujeres.

In [ ]:
# =============================================================================
# MODELO BASELINE — NACIONAL HOMBRES
# =============================================================================
logger.info("═" * 60)
logger.info("MODELO BASELINE — NACIONAL HOMBRES")
logger.info("═" * 60)

resultado_nac_hombres = evaluar_baseline(
    serie_prophet=serie_nac_hombres[['ds', 'y']].copy(),
    nombre="Nacional — Hombres",
    graficar=True
)

In [ ]:
# =============================================================================
# MODELO BASELINE — NACIONAL MUJERES
# =============================================================================
logger.info("═" * 60)
logger.info("MODELO BASELINE — NACIONAL MUJERES")
logger.info("═" * 60)

resultado_nac_mujeres = evaluar_baseline(
    serie_prophet=serie_nac_mujeres[['ds', 'y']].copy(),
    nombre="Nacional — Mujeres",
    graficar=True
)

### 6.1 Comparativa Nacional por Sexo

In [ ]:
# Tabla comparativa Nacional por Sexo
comp_sexo = pd.DataFrame([
    resultado_nac_hombres['resumen'],
    resultado_nac_mujeres['resumen']
])
comp_sexo = comp_sexo[['nombre', 'rmse', 'mae', 'mape', 'mdape', 'rmse_train', 'mape_train']]
comp_sexo.columns = ['Modelo', 'RMSE (CV)', 'MAE (CV)', 'MAPE (CV) %', 'MDAPE (CV)', 
                      'RMSE (Train)', 'MAPE (Train) %']
comp_sexo = comp_sexo.round(2)
print("\n" + "=" * 80)
print("COMPARATIVA BASELINE — NACIONAL POR SEXO")
print("=" * 80)
display(comp_sexo)

---
## 7. Modelo Baseline — Por Entidad Federativa <a id='7-modelo-baseline--por-entidad-federativa'></a>

Se entrena un modelo Prophet baseline independiente para cada una de las **32 entidades federativas**. Esta granularidad fue solicitada específicamente por las doctoras del IMSS para la asignación de recursos a nivel estatal.

> **Nota**: Los gráficos se muestran solo para un subconjunto representativo de estados para mantener el documento manejable. Las métricas se calculan para **todos** los estados.

In [ ]:
# =============================================================================
# MODELOS BASELINE — POR ENTIDAD FEDERATIVA (32 ESTADOS)
# =============================================================================
logger.info("═" * 60)
logger.info("MODELOS BASELINE — POR ENTIDAD FEDERATIVA")
logger.info("═" * 60)

# Estados representativos para gráficos detallados
ESTADOS_GRAFICOS = [
    "Ciudad de México", "Estado de México", "Jalisco", 
    "Nuevo León", "Aguascalientes", "Oaxaca"
]

resultados_estados = []

for i, estado in enumerate(entidades, 1):
    logger.info(f"[{i:02d}/{len(entidades)}] Entrenando: {estado}")
    
    serie_edo = serie_estados.loc[
        serie_estados['Entidad'] == estado, ['ds', 'y']
    ].copy()
    
    # Solo graficamos estados representativos
    mostrar_graficos = estado in ESTADOS_GRAFICOS
    
    try:
        resultado = evaluar_baseline(
            serie_prophet=serie_edo,
            nombre=f"Estado: {estado}",
            graficar=mostrar_graficos,
            verbose=True
        )
        resultados_estados.append(resultado['resumen'])
    except Exception as e:
        logger.error(f"Error en {estado}: {e}")
        resultados_estados.append({
            'nombre': f"Estado: {estado}",
            'rmse': np.nan, 'mae': np.nan, 'mape': np.nan, 'mdape': np.nan,
            'rmse_train': np.nan, 'mae_train': np.nan, 'mape_train': np.nan
        })

logger.success(f"Completados {len(resultados_estados)} modelos por estado")

In [ ]:
# =============================================================================
# TABLA DE RESULTADOS — TODAS LAS ENTIDADES
# =============================================================================
df_metricas_estados = pd.DataFrame(resultados_estados)
df_metricas_estados = df_metricas_estados[[
    'nombre', 'rmse', 'mae', 'mape', 'mdape', 'rmse_train', 'mape_train'
]].copy()
df_metricas_estados.columns = [
    'Entidad', 'RMSE (CV)', 'MAE (CV)', 'MAPE (CV) %', 'MDAPE (CV)',
    'RMSE (Train)', 'MAPE (Train) %'
]
df_metricas_estados = df_metricas_estados.sort_values('MAPE (CV) %').round(2)

print("\n" + "=" * 100)
print("MÉTRICAS BASELINE POR ENTIDAD FEDERATIVA (ordenado por MAPE)")
print("=" * 100)
display(df_metricas_estados.reset_index(drop=True))

In [ ]:
# =============================================================================
# VISUALIZACIÓN: MAPE POR ENTIDAD (BARRAS HORIZONTALES)
# =============================================================================
fig, ax = plt.subplots(figsize=(10, 12))

df_plot = df_metricas_estados.sort_values('MAPE (CV) %', ascending=True)

# Colorear según desempeño
colores = []
for mape in df_plot['MAPE (CV) %']:
    if mape <= 30:
        colores.append(IMSS_COLORS["teal"])
    elif mape <= 50:
        colores.append(IMSS_COLORS["gold"])
    else:
        colores.append(IMSS_COLORS["burgundy"])

ax.barh(df_plot['Entidad'].str.replace('Estado: ', ''), df_plot['MAPE (CV) %'], color=colores)
ax.set_xlabel('MAPE (%)', fontsize=12)
ax.set_title('MAPE del Modelo Baseline por Entidad Federativa\n(Cross-Validation)', 
             fontsize=13, fontweight='bold')
ax.axvline(x=30, color=IMSS_COLORS["teal"], linestyle='--', alpha=0.7, label='Buen desempeño (≤30%)')
ax.axvline(x=50, color=IMSS_COLORS["burgundy"], linestyle='--', alpha=0.7, label='Desempeño bajo (>50%)')
ax.legend(loc='lower right')
ax.grid(True, axis='x', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()
plt.close(fig)

---
## 8. Modelo Baseline — Por Entidad y Sexo <a id='8-modelo-baseline--por-entidad-y-sexo'></a>

Se ejecutan modelos desagregados por sexo para cada entidad. Esto permite evaluar si existen diferencias significativas en el desempeño predictivo entre hombres y mujeres a nivel estatal.

In [ ]:
# =============================================================================
# MODELOS BASELINE — POR ENTIDAD × SEXO
# =============================================================================
logger.info("═" * 60)
logger.info("MODELOS BASELINE — POR ENTIDAD × SEXO")
logger.info("═" * 60)

resultados_estado_sexo = []

for i, estado in enumerate(entidades, 1):
    for sexo, df_sexo in [('Hombres', serie_estados_hombres), ('Mujeres', serie_estados_mujeres)]:
        nombre_modelo = f"{estado} — {sexo}"
        
        serie_es = df_sexo.loc[
            df_sexo['Entidad'] == estado, ['ds', 'y']
        ].copy()
        
        try:
            resultado = evaluar_baseline(
                serie_prophet=serie_es,
                nombre=nombre_modelo,
                graficar=False,  # Sin gráficos para 64 modelos
                verbose=False
            )
            res = resultado['resumen']
            res['estado'] = estado
            res['sexo'] = sexo
            resultados_estado_sexo.append(res)
        except Exception as e:
            logger.warning(f"  ⚠ Error en {nombre_modelo}: {e}")
            resultados_estado_sexo.append({
                'nombre': nombre_modelo, 'estado': estado, 'sexo': sexo,
                'rmse': np.nan, 'mae': np.nan, 'mape': np.nan, 'mdape': np.nan,
                'rmse_train': np.nan, 'mae_train': np.nan, 'mape_train': np.nan
            })
    
    if i % 8 == 0:
        logger.info(f"  Progreso: {i}/{len(entidades)} estados completados")

logger.success(f"Completados {len(resultados_estado_sexo)} modelos (Estado × Sexo)")

In [ ]:
# =============================================================================
# TABLA RESUMEN — ESTADO × SEXO
# =============================================================================
df_estado_sexo = pd.DataFrame(resultados_estado_sexo)

# Pivot para comparar Hombres vs Mujeres
pivot_mape = df_estado_sexo.pivot_table(
    index='estado', columns='sexo', values='mape', aggfunc='first'
).round(2)

pivot_mape['Diferencia (M-H)'] = (pivot_mape['Mujeres'] - pivot_mape['Hombres']).round(2)
pivot_mape = pivot_mape.sort_values('Diferencia (M-H)', ascending=False)

print("\n" + "=" * 80)
print("MAPE (%) POR ENTIDAD Y SEXO — COMPARATIVA")
print("=" * 80)
display(pivot_mape)

In [ ]:
# =============================================================================
# VISUALIZACIÓN: COMPARATIVA MAPE HOMBRES VS MUJERES
# =============================================================================
fig, ax = plt.subplots(figsize=(12, 8))

x = np.arange(len(pivot_mape))
ancho = 0.35

bars_h = ax.barh(x + ancho/2, pivot_mape['Hombres'], ancho, 
                 label='Hombres', color=IMSS_COLORS["teal"], alpha=0.85)
bars_m = ax.barh(x - ancho/2, pivot_mape['Mujeres'], ancho, 
                 label='Mujeres', color=IMSS_COLORS["burgundy"], alpha=0.85)

ax.set_yticks(x)
ax.set_yticklabels(pivot_mape.index, fontsize=9)
ax.set_xlabel('MAPE (%)')
ax.set_title('MAPE Baseline: Hombres vs. Mujeres por Entidad\n(Cross-Validation)', 
             fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, axis='x', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()
plt.close(fig)

---
## 9. Análisis de Sub/Sobreajuste <a id='9-análisis-de-subsobreajuste'></a>

Para evaluar si el modelo baseline presenta **subajuste** (*underfitting*) o **sobreajuste** (*overfitting*), se comparan las métricas de desempeño en el conjunto de **entrenamiento** frente al de **validación cruzada** (Géron, 2022).

| Condición | Indicador | Acción |
|---|---|---|
| **Subajuste** | Error alto en train Y en CV | El modelo es demasiado simple; considerar hiperparámetros más flexibles |
| **Buen ajuste** | Error bajo en train, similar en CV | El modelo generaliza correctamente |
| **Sobreajuste** | Error bajo en train, alto en CV | El modelo memoriza; considerar regularización |

### 9.1 Comparativa Train vs. Cross-Validation — Nivel Nacional

In [ ]:
# =============================================================================
# ANÁLISIS SUB/SOBREAJUSTE — NIVEL NACIONAL
# =============================================================================
modelos_nacionales = [
    resultado_nacional['resumen'],
    resultado_nac_hombres['resumen'],
    resultado_nac_mujeres['resumen']
]

df_ajuste = pd.DataFrame(modelos_nacionales)[[
    'nombre', 'mape_train', 'mape', 'rmse_train', 'rmse'
]].copy()
df_ajuste.columns = ['Modelo', 'MAPE Train %', 'MAPE CV %', 'RMSE Train', 'RMSE CV']
df_ajuste['Brecha MAPE (CV-Train)'] = (df_ajuste['MAPE CV %'] - df_ajuste['MAPE Train %']).round(2)
df_ajuste['Brecha RMSE (CV-Train)'] = (df_ajuste['RMSE CV'] - df_ajuste['RMSE Train']).round(2)
df_ajuste = df_ajuste.round(2)

print("\n" + "=" * 90)
print("ANÁLISIS DE SUB/SOBREAJUSTE — MODELOS NACIONALES")
print("=" * 90)
display(df_ajuste)

print("\nInterpretación:")
for _, row in df_ajuste.iterrows():
    brecha = row['Brecha MAPE (CV-Train)']
    if brecha > 15:
        print(f"  ⚠ {row['Modelo']}: Posible SOBREAJUSTE (brecha MAPE = {brecha}%)")
    elif row['MAPE Train %'] > 40:
        print(f"  ⚠ {row['Modelo']}: Posible SUBAJUSTE (MAPE train = {row['MAPE Train %']}%)")
    else:
        print(f"  ✓ {row['Modelo']}: Ajuste ACEPTABLE (brecha MAPE = {brecha}%)")

In [ ]:
# =============================================================================
# VISUALIZACIÓN: BRECHAS TRAIN vs. CV POR ESTADO
# =============================================================================
df_estados_ajuste = pd.DataFrame(resultados_estados)[[
    'nombre', 'mape_train', 'mape'
]].dropna()
df_estados_ajuste.columns = ['Entidad', 'MAPE Train', 'MAPE CV']
df_estados_ajuste['Brecha'] = df_estados_ajuste['MAPE CV'] - df_estados_ajuste['MAPE Train']
df_estados_ajuste = df_estados_ajuste.sort_values('Brecha', ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))

y_pos = np.arange(len(df_estados_ajuste))
ax.scatter(df_estados_ajuste['MAPE Train'], y_pos, color=IMSS_COLORS['teal'], 
           s=60, zorder=3, label='Train')
ax.scatter(df_estados_ajuste['MAPE CV'], y_pos, color=IMSS_COLORS['burgundy'], 
           s=60, zorder=3, label='Cross-Validation')

for i, (_, row) in enumerate(df_estados_ajuste.iterrows()):
    ax.plot([row['MAPE Train'], row['MAPE CV']], [i, i], 
            color=IMSS_COLORS['cool_gray'], linewidth=1, alpha=0.6)

ax.set_yticks(y_pos)
ax.set_yticklabels(df_estados_ajuste['Entidad'].str.replace('Estado: ', ''), fontsize=8)
ax.set_xlabel('MAPE (%)')
ax.set_title('Análisis de Sub/Sobreajuste por Entidad\n(Brecha entre Train y CV)', 
             fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, axis='x', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()
plt.close(fig)

---
## 10. Análisis de Componentes (Importancia de Características) <a id='10-análisis-de-componentes-importancia-de-características'></a>

En el contexto de Prophet, el análisis de **importancia de características** se traduce en el análisis de los **componentes del modelo** (Taylor & Letham, 2018):

- **Tendencia** (*trend*): Captura el comportamiento a largo plazo. Su magnitud y dirección reflejan la evolución estructural de la incidencia.
- **Estacionalidad anual** (*yearly seasonality*): Cuantifica los patrones cíclicos anuales. Su amplitud indica qué tan pronunciados son los ciclos estacionales.
- **Changepoints**: Prophet detecta automáticamente puntos de cambio en la tendencia. Su ubicación temporal es fundamental para entender quiebres estructurales (e.g., inicio del confinamiento COVID-19).

> **Nota**: A diferencia de modelos supervisados tradicionales (Random Forest, XGBoost) donde la importancia se asigna a *features* explícitas, Prophet opera exclusivamente con la variable temporal. Las "características" relevantes son los componentes descompuestos de la serie.

### 10.1 Contribución Relativa de Componentes

In [ ]:
# =============================================================================
# ANÁLISIS DE COMPONENTES — MODELO NACIONAL
# =============================================================================
forecast_nac = resultado_nacional['forecast']

# Calcular contribución relativa de cada componente
componentes = ['trend', 'yearly']
cols_disponibles = [c for c in componentes if c in forecast_nac.columns]

if cols_disponibles:
    # Varianza explicada por cada componente
    varianzas = {}
    for comp in cols_disponibles:
        varianzas[comp] = forecast_nac[comp].var()
    
    total_var = sum(varianzas.values())
    print("\n" + "=" * 60)
    print("CONTRIBUCIÓN RELATIVA DE COMPONENTES — MODELO NACIONAL")
    print("=" * 60)
    for comp, var in varianzas.items():
        pct = (var / total_var) * 100
        print(f"  {comp.capitalize():20s} → Varianza: {var:12.2f} ({pct:.1f}%)")
    
    # Gráfico de barras
    fig, ax = plt.subplots(figsize=(8, 4))
    nombres = [c.capitalize() for c in cols_disponibles]
    valores = [(varianzas[c] / total_var) * 100 for c in cols_disponibles]
    colores_comp = [IMSS_COLORS['teal'], IMSS_COLORS['burgundy'], IMSS_COLORS['gold']][:len(cols_disponibles)]
    
    ax.bar(nombres, valores, color=colores_comp)
    ax.set_ylabel('Varianza Explicada (%)')
    ax.set_title('Contribución Relativa de Componentes — Modelo Nacional Baseline',
                 fontsize=12, fontweight='bold')
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)
    
    for i, v in enumerate(valores):
        ax.text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    plt.close(fig)

In [ ]:
# =============================================================================
# CHANGEPOINTS DETECTADOS
# =============================================================================
modelo_nac = resultado_nacional['modelo']

print("\n" + "=" * 60)
print("CHANGEPOINTS DETECTADOS — MODELO NACIONAL")
print("=" * 60)
print(f"Número de changepoints: {len(modelo_nac.changepoints)}")
print(f"\nPrimeros 5 changepoints:")
for i, cp in enumerate(modelo_nac.changepoints[:5]):
    print(f"  {i+1}. {cp.strftime('%Y-%m-%d')}")
print(f"\nÚltimos 5 changepoints:")
for i, cp in enumerate(modelo_nac.changepoints[-5:]):
    print(f"  {len(modelo_nac.changepoints)-4+i}. {cp.strftime('%Y-%m-%d')}")

# Verificar si COVID-19 fue detectado como changepoint
covid_cps = [cp for cp in modelo_nac.changepoints 
             if pd.Timestamp('2020-01-01') <= cp <= pd.Timestamp('2020-12-31')]
if covid_cps:
    print(f"\n✓ Se detectaron {len(covid_cps)} changepoint(s) durante 2020 (período COVID-19):")
    for cp in covid_cps:
        print(f"  → {cp.strftime('%Y-%m-%d')}")
else:
    print("\n⚠ No se detectaron changepoints explícitos durante 2020")

---
## 11. Resumen Consolidado de Métricas <a id='11-resumen-consolidado-de-métricas'></a>

Se consolidan las métricas de todos los niveles de modelado en una vista unificada.

In [ ]:
# =============================================================================
# RESUMEN CONSOLIDADO — TODOS LOS NIVELES
# =============================================================================

# Estadísticas agregadas de los modelos por estado
df_est = pd.DataFrame(resultados_estados)
stats_estados = {
    'nombre': 'Estados (Promedio de 32)',
    'rmse': df_est['rmse'].mean(),
    'mae': df_est['mae'].mean(),
    'mape': df_est['mape'].mean(),
    'mdape': df_est['mdape'].mean() if 'mdape' in df_est.columns else np.nan,
    'rmse_train': df_est['rmse_train'].mean(),
    'mape_train': df_est['mape_train'].mean(),
}

# Estadísticas agregadas de modelos estado×sexo
df_es = pd.DataFrame(resultados_estado_sexo)
stats_estado_sexo = {
    'nombre': 'Estado×Sexo (Promedio de 64)',
    'rmse': df_es['rmse'].mean(),
    'mae': df_es['mae'].mean(),
    'mape': df_es['mape'].mean(),
    'mdape': df_es['mdape'].mean() if 'mdape' in df_es.columns else np.nan,
    'rmse_train': df_es['rmse_train'].mean(),
    'mape_train': df_es['mape_train'].mean(),
}

# Tabla consolidada
resumen_todos = pd.DataFrame([
    resultado_nacional['resumen'],
    resultado_nac_hombres['resumen'],
    resultado_nac_mujeres['resumen'],
    stats_estados,
    stats_estado_sexo
])[['nombre', 'rmse', 'mae', 'mape', 'mdape', 'rmse_train', 'mape_train']]

resumen_todos.columns = [
    'Nivel de Modelo', 'RMSE (CV)', 'MAE (CV)', 'MAPE (CV) %', 'MDAPE (CV)',
    'RMSE (Train)', 'MAPE (Train) %'
]
resumen_todos = resumen_todos.round(2)

print("\n" + "═" * 100)
print("RESUMEN CONSOLIDADO — MODELO BASELINE EPIFORECAST-MX")
print("═" * 100)
display(resumen_todos)
print("\nTotal de modelos entrenados:")
print(f"  • Nacional:        1 modelo")
print(f"  • Nacional×Sexo:   2 modelos")
print(f"  • Por Estado:      {len(resultados_estados)} modelos")
print(f"  • Estado×Sexo:     {len(resultados_estado_sexo)} modelos")
print(f"  • TOTAL:           {1 + 2 + len(resultados_estados) + len(resultados_estado_sexo)} modelos baseline")

---
## 12. Desempeño Mínimo Aceptable <a id='12-desempeño-mínimo-aceptable'></a>

### 12.1 Definición del Umbral

No existe un antecedente histórico directo de pronóstico epidemiológico automatizado para estos tres padecimientos en el contexto del IMSS. Por lo tanto, el desempeño mínimo se establece con base en:

1. **Criterios de la literatura**: En pronóstico de series de tiempo epidemiológicas, un MAPE inferior al 30% se considera aceptable para planificación estratégica (Hyndman & Athanasopoulos, 2021).
2. **Requisitos institucionales**: La Dra. Ruth enfatizó que los pronósticos deben ser útiles para la **asignación de recursos** y **planificación presupuestaria**, lo cual requiere un nivel de precisión que permita tomar decisiones informadas.
3. **Desempeño del baseline**: El modelo actual establece el piso contra el cual los modelos tuneados deberán mejorar.

### 12.2 Umbrales Propuestos

In [ ]:
# =============================================================================
# VERIFICACIÓN DE DESEMPEÑO MÍNIMO
# =============================================================================
UMBRAL_MAPE_EXCELENTE = 20   # Excelente: ≤ 20%
UMBRAL_MAPE_ACEPTABLE = 30   # Aceptable: ≤ 30%
UMBRAL_MAPE_LIMITE    = 50   # Límite: ≤ 50% (requiere mejora urgente)

print("\n" + "═" * 70)
print("UMBRALES DE DESEMPEÑO MÍNIMO ACEPTABLE")
print("═" * 70)
print(f"  🟢 Excelente:  MAPE ≤ {UMBRAL_MAPE_EXCELENTE}%")
print(f"  🟡 Aceptable:  MAPE ≤ {UMBRAL_MAPE_ACEPTABLE}%")
print(f"  🟠 Requiere mejora: MAPE ≤ {UMBRAL_MAPE_LIMITE}%")
print(f"  🔴 Desempeño insuficiente: MAPE > {UMBRAL_MAPE_LIMITE}%")

# Clasificar estados
df_clasificacion = pd.DataFrame(resultados_estados)[['nombre', 'mape']].dropna()
df_clasificacion['Clasificación'] = pd.cut(
    df_clasificacion['mape'],
    bins=[0, UMBRAL_MAPE_EXCELENTE, UMBRAL_MAPE_ACEPTABLE, UMBRAL_MAPE_LIMITE, float('inf')],
    labels=['🟢 Excelente', '🟡 Aceptable', '🟠 Requiere mejora', '🔴 Insuficiente']
)

conteo = df_clasificacion['Clasificación'].value_counts().sort_index()
print("\nDistribución de estados por nivel de desempeño:")
for cat, n in conteo.items():
    print(f"  {cat}: {n} estados")

# Verificación del modelo nacional
mape_nac = resultado_nacional['resumen']['mape']
print(f"\nModelo Nacional → MAPE: {mape_nac:.2f}%", end=" ")
if mape_nac <= UMBRAL_MAPE_ACEPTABLE:
    print("✅ CUMPLE el desempeño mínimo aceptable")
else:
    print("⚠️ REQUIERE optimización mediante tuning de hiperparámetros")

---
## 13. Conclusiones y Siguientes Pasos <a id='13-conclusiones-y-siguientes-pasos'></a>

### 13.1 Hallazgos Principales

1. **Viabilidad confirmada**: El modelo baseline demuestra que Prophet puede capturar la dinámica temporal de los padecimientos neurológicos y de salud mental reportados en el boletín epidemiológico de SINAVE. El desempeño obtenido supera al de un modelo naïve, confirmando que los datos contienen información predictiva.

2. **Descomposición interpretable**: Los componentes de tendencia y estacionalidad anual son interpretables y consistentes con el conocimiento epidemiológico: se observan patrones estacionales y el efecto disruptivo de la pandemia COVID-19 se captura mediante changepoints.

3. **Variabilidad por entidad**: Existe una dispersión significativa en el desempeño entre estados, lo cual valida la decisión de entrenar modelos individuales por entidad en lugar de un modelo único. Los estados con bajo reporte (pocos casos semanales) presentan mayor incertidumbre, como lo anticipó la Dra. Grettel.

4. **Diferencias por sexo**: Los modelos desagregados por sexo permiten identificar diferencias en la predecibilidad entre hombres y mujeres, información relevante para el prorrateo proporcional solicitado por las doctoras del IMSS.

### 13.2 Línea de Mejora: Del Baseline al Modelo Tuneado

El baseline utiliza hiperparámetros por defecto de Prophet. Las siguientes optimizaciones se explorarán en el **Avance 4** (modelos tuneados):

| Hiperparámetro | Valor Baseline | Exploración Planificada |
|---|---|---|
| `seasonality_mode` | `additive` | `multiplicative` (para estados con estacionalidad proporcional a la tendencia) |
| `changepoint_prior_scale` | 0.05 | Grid search: [0.01, 0.05, 0.1, 0.5, 1.0] |
| `seasonality_prior_scale` | 10.0 | Grid search: [0.1, 0.5, 1.0, 5.0, 10.0] |
| `holidays` | ninguno | Incorporar días festivos mexicanos y periodos vacacionales |
| `regressors` | ninguno | Explorar variables exógenas (densidad poblacional, región socioeconómica) |

### 13.3 Alineación con Objetivos Estratégicos del Proyecto

Este baseline contribuye a los tres objetivos estratégicos definidos por la Dra. Ruth:

- **Académico**: Cumple con los requisitos del Módulo 3 del curso TC5035 (métricas de desempeño + modelo de referencia).
- **Científico**: Establece la línea base reproducible para los artículos en desarrollo, siguiendo la estructura metodológica recomendada.
- **Operativo**: Los pronósticos baseline ya son potencialmente útiles para planificación inicial; los modelos tuneados mejorarán la precisión para uso institucional.

---
## 14. Referencias <a id='14-referencias'></a>

- Costa, R. (2022). *The CRISP-ML Methodology: A Step-by-Step Approach to Real-World Machine Learning Projects*. Edición propia.

- Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3ra ed.). O'Reilly Media.

- Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice* (3ra ed.). OTexts. https://otexts.com/fpp3/

- Hyndman, R. J., & Koehler, A. B. (2006). Another look at measures of forecast accuracy. *International Journal of Forecasting*, 22(4), 679–688. https://doi.org/10.1016/j.ijforecast.2006.03.001

- Piccini, N. (2023, julio 19). 101 machine learning algorithms for data science with cheat sheets. *Data Science Dojo*. https://datasciencedojo.com/blog/machine-learning-algorithms/

- Studer, S., Bui, T. B., Drescher, C., Hanuschkin, A., Winkler, L., Peters, S., & Müller, K.-R. (2021). Towards CRISP-ML(Q): A Machine Learning Process Model with Quality Assurance Methodology. *Preprints*, 2021, 1, 0. https://doi.org/10.48550/arXiv.2003.05155

- Taylor, S. J., & Letham, B. (2018). Forecasting at scale. *The American Statistician*, 72(1), 37–45. https://doi.org/10.1080/00031305.2017.1380080

- Visengeriyeva, L., Kammer, A., Bär, I., Kniesz, A., & Plöd, M. (2023). CRISP-ML(Q). *The ML Lifecycle Process. MLOps. INNOQ*. https://ml-ops.org/content/crisp-ml

---

<center>

**EpiForecast-MX** · Proyecto en colaboración con el Instituto Mexicano del Seguro Social (IMSS)

Tecnológico de Monterrey · Maestría en Inteligencia Artificial Aplicada · 2026

</center>